# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")


## 2. Data Overview
Review available record sets, fields, and their IDs using the dataset's metadata.


In [ ]:
# List available record sets and their fields, referencing by @id
record_sets = metadata.record_sets
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '')}")
    if rs.get('fields'):
        print("    Fields:")
        for field in rs['fields']:
            if isinstance(field, dict):
                print(f"      - {field['@id']}: {field.get('name', '')}")
            else:
                print(f"      - {field}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
rs_ids = [rs['@id'] for rs in metadata.record_sets]
dfs = {}
for rs_id in rs_ids:
    try:
        print(f"Loading records from RecordSet {rs_id} ...")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"Columns for RecordSet {rs_id}:")
        print(df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records from {rs_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section applies to one record set for demonstration.

In [ ]:
# Choose the main protein record set (update with your actual @id if needed)

if len(dfs) > 0:
    main_record_set_id = rs_ids[0]
    main_df = dfs[main_record_set_id]

    print(f"Selected record set for EDA: {main_record_set_id}")

    # Find a numeric field to analyze
    # We'll try to infer a likely field name (e.g., one containing 'Abundance', 'MW', or 'Count')
    numeric_candidates = [
        col for col in main_df.columns 
        if any(key in col.lower() for key in ['abundance', 'mw', 'count', 'coverage', 'number'])
           and pd.api.types.is_numeric_dtype(main_df[col])
    ]
    print("Numeric candidate fields:", numeric_candidates)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        threshold = main_df[numeric_field].mean()
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records: {len(filtered_df)} with {numeric_field} > mean ({threshold:.3f})")
        
        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, norm_col]].head())
        # Attempt grouping by a non-numeric field, e.g. 'Sample' or 'Description'
        group_field = None
        non_num_fields = [col for col in main_df.columns if not pd.api.types.is_numeric_dtype(main_df[col])]
        for f in ['Sample', 'Description', 'Accession']:
            for col in non_num_fields:
                if f.lower() in col.lower():
                    group_field = col
                    break
            if group_field:
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame loaded; cannot perform EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dfs) > 0 and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # If grouping field is available, plot top groups
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,4))
        grouped_df.head(10).plot(kind='bar')
        plt.title(f"Top 10 {group_field}s by average {numeric_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using `mlcroissant`. We demonstrated how to:
- Discover record sets and their fields by their `@id`
- Extract records using these identifiers
- Perform basic exploratory data analysis (EDA) and group statistics
- Visualize field distributions

This template can be extended to other record sets, filtered by different field criteria, and used to build more advanced analyses and workflows.